In [1]:
# Cell 1: bootstrap, local model config, and browser profile.
from __future__ import annotations

import json
import os
import urllib.error
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path(r"D:\_Desktop\Projects\Automations prj\Job_search")
LOCAL_USER_DATA_DIR = Path(r"D:\_Desktop\Projects\Automations prj\User Data")
LLAMA_BASE_URL = os.environ.get("BROWSER_USE_LLM_BASE_URL", "http://127.0.0.1:8080/v1")
LLAMA_MODEL_FALLBACK = os.environ.get("BROWSER_USE_LLM_MODEL", "Qwen3.5-9B.Q4_K_M.gguf")
LLAMA_API_KEY = os.environ.get("BROWSER_USE_LLM_API_KEY", "sk-local")
MAX_HISTORY_ITEMS = 8
FLASH_MODE = True
USE_VISION = False

def _load_json(url: str, timeout: int = 10) -> dict:
    request = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def resolve_model_name() -> str:
    try:
        payload = _load_json(f"{LLAMA_BASE_URL.rstrip('/')}/models")
        models = payload.get("data") or []
        if models and isinstance(models, list):
            first = models[0]
            if isinstance(first, dict) and first.get("id"):
                return str(first["id"])
    except Exception as exc:
        print(f"Model probe failed, using fallback: {exc}")
    return LLAMA_MODEL_FALLBACK

MODEL_NAME = resolve_model_name()

print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "user_data_dir": str(LOCAL_USER_DATA_DIR),
    "llama_base_url": LLAMA_BASE_URL,
    "model": MODEL_NAME,
    "max_history_items": MAX_HISTORY_ITEMS,
    "flash_mode": FLASH_MODE,
    "use_vision": USE_VISION,
}, indent=2))


{
  "project_root": "D:\\_Desktop\\Projects\\Automations prj\\Job_search",
  "user_data_dir": "D:\\_Desktop\\Projects\\Automations prj\\User Data",
  "llama_base_url": "http://127.0.0.1:8080/v1",
  "model": "Qwen3.5-9B.Q4_K_M.gguf",
  "max_history_items": 8,
  "flash_mode": true,
  "use_vision": false
}


In [2]:
# Cell 2: Browser Use agent setup and runner.
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm.openai.chat import ChatOpenAI

browser_profile = BrowserProfile(
    user_data_dir=str(LOCAL_USER_DATA_DIR),
    headless=False,
    window_size={"width": 1440, "height": 1080},
    wait_between_actions=1.0,
    minimum_wait_page_load_time=0.5,
    wait_for_network_idle_page_load_time=0.8,
    highlight_elements=True,
)

browser_session = BrowserSession(browser_profile=browser_profile)

llm = ChatOpenAI(
    model=MODEL_NAME,
    base_url=LLAMA_BASE_URL,
    api_key=LLAMA_API_KEY,
    temperature=0,
    max_completion_tokens=8192,
    reasoning_effort="low",
    add_schema_to_system_prompt=True,
    dont_force_structured_output=True,
    remove_min_items_from_schema=True,
    remove_defaults_from_schema=True,
)

async def run_task(task: str, max_steps: int = 20):
    agent = Agent(
        task=task,
        llm=llm,
        browser_session=browser_session,
        browser_profile=browser_profile,
        use_vision=USE_VISION,
        flash_mode=FLASH_MODE,
        max_history_items=MAX_HISTORY_ITEMS,
        enable_planning=False,
        directly_open_url=True,
        include_recent_events=True,
        llm_timeout=120,
        step_timeout=180,
        max_actions_per_step=5,
        use_thinking=True,
    )
    history = await agent.run(max_steps=max_steps)
    print(json.dumps({
        "done": history.is_done(),
        "successful": history.is_successful(),
        "steps": history.number_of_steps(),
    }, indent=2))
    try:
        print("final_result:")
        print(history.final_result())
    except Exception as exc:
        print(f"final_result unavailable: {exc}")
    return history


In [3]:
# Cell 3: Edit TASK and run the browser-use agent.
TASK = "Go to https://duckduckgo.com and report the page title."
history = await run_task(TASK, max_steps=12)
history


INFO     [service] Using anonymized telemetry, see https://docs.browser-use.com/development/monitoring/telemetry.
INFO     [Agent] 🎯 Task: Search DuckDuckGo for the cheapest VPS server available and return the best result.
INFO     [Agent] Starting a browser-use agent with version 0.13.1, with provider=openai and model=Qwen3.5-9B.Q4_K_M.gguf
INFO     [utils] 📦 Downloading uBlock Origin Lite extension...
INFO     [utils] 📂 Extracting uBlock Origin Lite extension...
INFO     [utils] 📦 Downloading I still don't care about cookies extension...
INFO     [utils] 📂 Extracting I still don't care about cookies extension...
INFO     [utils] 📦 Downloading Force Background Tab extension...
INFO     [utils] 📂 Extracting Force Background Tab extension...
INFO     [utils] [BrowserProfile] ✅ Cookie extension: nature.com, qatarairways.com pre-populated in storage
ERROR    [BrowserSession] [LocalBrowserWatchdog] Exception in on_BrowserLaunchEvent: 
Traceback (most recent call last):
  File "d:\Conda\env

NotImplementedError: 